# RSM8542 — Internal Liquidity Marketplace: Real Data Experiment

**Four matching engines are compared against a common ground truth:**

| # | Engine | Method |
|---|--------|--------|
| — | **Baseline** | TF-IDF cosine similarity on resume text only |
| A | **Claude Only** | TF-IDF pre-filter (top-20) → Claude re-ranks full profiles |
| B | **Embeddings Only** | OpenAI `text-embedding-3-large` on full profiles → top-3 by cosine sim |
| C | **Shortlist + Claude** | Embedding pre-filter (top-20) → Claude re-ranks full profiles |

**Pipeline**
1. Load datasets (Jira, GitHub commits, job postings, resumes)
2. Filter 100 technical resumes and build composite profiles (resume + Jira + commits)
3. Sample 20 job postings
4. Run Baseline
5. Pre-compute OpenAI embeddings for all 100 profiles (once, reused by B and C)
6. Run Engine A — Claude Only
7. Run Engine B — Embeddings Only
8. Run Engine C — Shortlist + Claude
9. Generate ground truth (Claude as hiring manager, resume-only view, cached)
10. Compute Precision@3 and lift for all four engines


In [1]:
# ── Install dependencies ───────────────────────────────────────────────────────
# Run once in your environment before executing the notebook
!pip install kagglehub[pandas-datasets] pandas openpyxl scikit-learn anthropic openai tiktoken tqdm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 635.9/635.9 kB 20.2 MB/s eta 0:00:00


In [4]:
# ── Cell 1: Imports ────────────────────────────────────────────────────────────
import os, json, random, re, time
import numpy as np
import pandas as pd
import tiktoken
import kagglehub
from kagglehub import KaggleDatasetAdapter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from openai import OpenAI
import anthropic
from tqdm import tqdm

# ── API keys — set in your shell before running ────────────────────────────────
#   export ANTHROPIC_API_KEY="sk-ant-..."
#   export OPENAI_API_KEY="sk-proj-..."

anthropic_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
openai_client    = OpenAI(api_key=OPENAI_API_KEY)

random.seed(42)
np.random.seed(42)
print("Imports OK")


Imports OK


In [5]:
# ── CELL 2: Configuration ──────────────────────────────────────────────────────

# Sampling targets
N_RESUMES            = 100
N_JIRA               = 500
N_COMMITS            = 500
N_POSTINGS           = 20
TICKETS_PER_PROFILE  = 5    # N jira tickets randomly assigned to each resume
COMMITS_PER_PROFILE  = 10   # M commits randomly assigned to each resume
JIRA_MIN_DESC_LEN    = 80   # minimum character length to include a jira ticket
TOP_K                = 3    # Precision@k

# Resume dataset column names — update if your xlsx uses different names
RESUME_TEXT_COL      = "Resume_str"
RESUME_CATEGORY_COL  = "Category"

# GitHub commits column names (from dhruvildave dataset)
COMMIT_MSG_COL       = "message"
COMMIT_REPO_COL      = "repo"

# Job postings column names (from adityarajsrv dataset)
# Inspect job_descriptions.columns after loading and update if needed
JOB_TITLE_COL        = "Title"
JOB_DESC_COL         = "Job Description"

# Jira columns
JIRA_SUMMARY_COL     = "Summary"
JIRA_DESC_COL        = "Description"

# Technical role keywords used to filter resumes and match jira/commits
TECHNICAL_KEYWORDS = [
    "software", "engineer", "developer", "data", "analyst", "machine learning",
    "devops", "backend", "frontend", "python", "java", "cloud", "infrastructure",
    "security", "database", "architect", "scientist", "ml", "ai", "nlp",
    "deep learning", "api", "kubernetes", "docker", "spark", "sql", "etl"
]

In [6]:
# ── CELL 3: Load raw data ──────────────────────────────────────────────────────

# Jira tickets
jira_raw = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "cesaranasco/jira-dataset-public",
    "GFG_FINAL.csv"
)
jira_raw = pd.DataFrame(jira_raw, columns=["Summary", "Description"])
print(f"Jira raw: {len(jira_raw)} rows")

# Job postings
job_descriptions = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "adityarajsrv/job-descriptions-2025-tech-and-non-tech-roles",
    "job_dataset.csv"
)
print(f"Job postings raw: {len(job_descriptions)} rows")
print(f"Job posting columns: {job_descriptions.columns.tolist()}")

# GitHub commits
path = kagglehub.dataset_download("dhruvildave/github-commit-messages-dataset")
commit_file = "/root/.cache/kagglehub/datasets/dhruvildave/github-commit-messages-dataset/versions/3/full.csv"
github_raw = pd.read_csv(commit_file)
print(f"GitHub raw: {len(github_raw)} rows")
print(f"GitHub columns: {github_raw.columns.tolist()}")

/tmp/ipykernel_8616/1243390484.py:4: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  jira_raw = kagglehub.load_dataset(


100%|██████████| 24.2M/24.2M [00:00<00:00, 152MB/s]

Extracting zip of GFG_FINAL.csv...


Jira raw: 49000 rows


/tmp/ipykernel_8616/1243390484.py:13: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  job_descriptions = kagglehub.load_dataset(


100%|██████████| 597k/597k [00:00<00:00, 10.9MB/s]

Job postings raw: 1068 rows
Job posting columns: ['JobID', 'Title', 'ExperienceLevel', 'YearsOfExperience', 'Skills', 'Responsibilities', 'Keywords']


100%|██████████| 903M/903M [00:05<00:00, 174MB/s]

Extracting files...


GitHub raw: 4336299 rows
GitHub columns: ['commit', 'author', 'date', 'message', 'repo']


In [7]:
job_descriptions["Job Description"] =  job_descriptions['Skills'].astype(str) + " " + job_descriptions['Responsibilities'].astype(str)


In [8]:

# Resumes
resumes_raw = pd.read_excel("resume.xlsx")
print(f"Resumes raw: {len(resumes_raw)} rows")
print(f"Resume columns: {resumes_raw.columns.tolist()}")

Resumes raw: 119 rows
Resume columns: ['ID', 'Resume_str']


In [9]:
# ── CELL 4: Inspect samples — verify column names match config above ───────────
print("=== JIRA SAMPLE ===")
display(jira_raw.head(3))

print("\n=== JOB POSTINGS SAMPLE ===")
display(job_descriptions.head(3))

print("\n=== GITHUB COMMITS SAMPLE ===")
display(github_raw.head(3))

print("\n=== RESUMES SAMPLE ===")
display(resumes_raw.head(3))

=== JIRA SAMPLE ===


,Summary,Description
0,Authentication failed when attempting Fetch c...,Receiving following error when attempting Fetc...
1,OAuth token keeps expiring,I refreshed my OAuth token just yesterday and ...
2,Mercurial: No column for revision/changeset nu...,"By default, Sourcetree 3.4.9 displays a ""Commi..."



=== JOB POSTINGS SAMPLE ===


,JobID,Title,ExperienceLevel,YearsOfExperience,Skills,Responsibilities,Keywords,Job Description
0,NET-F-001,.NET Developer,Fresher,0-1,C#; VB.NET basics; .NET Framework; .NET Core f...,Assist in coding and debugging applications; L...,.NET; C#; ASP.NET MVC; Entity Framework; SQL S...,C#; VB.NET basics; .NET Framework; .NET Core f...
1,NET-F-002,.NET Developer,Fresher,0-1,C#; .NET Framework basics; ASP.NET; Razor; HTM...,Write simple C# programs under guidance; Suppo...,.NET; C#; ASP.NET MVC; Entity Framework; SQL S...,C#; .NET Framework basics; ASP.NET; Razor; HTM...
2,NET-F-003,.NET Developer,Fresher,0-1,C#; VB.NET basics; .NET Core; ASP.NET MVC; HTM...,Contribute to development of small modules; As...,.NET; C#; ASP.NET MVC; SQL Server; Entity Fram...,C#; VB.NET basics; .NET Core; ASP.NET MVC; HTM...



=== GITHUB COMMITS SAMPLE ===


,commit,author,date,message,repo
0,692bba578efb5e305c9b116568e5aad75b3fdbb3,Mortada Mehyar <mortada@users.noreply.github.com>,Wed Apr 21 12:27:07 2021 +0800,DOC: add example for plotting asymmetrical err...,pandas-dev/pandas
1,855696cde0ef5d80a7d4bd3f6a2940c5a2fecb3f,Patrick Hoefler <61934744+phofl@users.noreply....,Wed Apr 21 01:23:07 2021 +0200,Add keyword sort to pivot_table (#40954),pandas-dev/pandas
2,eaaefd140289a5103679ac6748567f724c7be56a,attack68 <24256554+attack68@users.noreply.gith...,Wed Apr 21 01:21:22 2021 +0200,ENH: `Styler.highlight_quantile` method (#40926),pandas-dev/pandas



=== RESUMES SAMPLE ===


,ID,Resume_str
0,36856210,INFORMATION TECHNOLOGY Summar...
1,21780877,INFORMATION TECHNOLOGY SPECIALIST\tGS...
2,33241454,INFORMATION TECHNOLOGY SUPERVISOR ...


In [10]:
# ── CELL 5: Helper — keyword match score ──────────────────────────────────────
# Counts how many technical keywords appear in a piece of text.
# Used to rank Jira tickets and commits before sampling.

def keyword_score(text, keywords):
    if not isinstance(text, str):
        return 0
    text_lower = text.lower()
    return sum(1 for kw in keywords if kw in text_lower)

def is_technical_resume(text, keywords, threshold=3):
    return keyword_score(text, keywords) >= threshold

In [11]:
# ── CELL 6: Filter 100 technical resumes ──────────────────────────────────────

resumes_raw["tech_score"] = resumes_raw[RESUME_TEXT_COL].apply(
    lambda x: keyword_score(x, TECHNICAL_KEYWORDS)
)

# Keep rows that pass the technical threshold, then sample 100
resumes_tech = resumes_raw[resumes_raw["tech_score"] >= 3].copy()
print(f"Technical resumes after filter: {len(resumes_tech)}")

if len(resumes_tech) < N_RESUMES:
    print(f"WARNING: only {len(resumes_tech)} technical resumes found, lower threshold or use all")
    resumes_sample = resumes_tech.reset_index(drop=True)
else:
    resumes_sample = resumes_tech.sample(n=N_RESUMES, random_state=42).reset_index(drop=True)

resumes_sample["employee_id"] = [f"E{i:03d}" for i in range(len(resumes_sample))]
print(f"Final resume sample: {len(resumes_sample)} employees")
display(resumes_sample[["employee_id", "tech_score"]].head(10))

Technical resumes after filter: 117
Final resume sample: 100 employees


,employee_id,tech_score
0,E000,5
1,E001,5
2,E002,7
3,E003,9
4,E004,7
5,E005,9
6,E006,8
7,E007,6
8,E008,8
9,E009,7


In [12]:
# ── CELL 7: Build resume vocabulary for Jira and commit matching ──────────────
# Concatenate all resume text into one blob, extract top TF-IDF terms.
# These terms are used to keyword-match Jira tickets and commits to resumes.

all_resume_text = " ".join(resumes_sample[RESUME_TEXT_COL].dropna().tolist())

# Extract top 200 terms from the resume corpus using TF-IDF
tfidf_vocab = TfidfVectorizer(
    stop_words="english",
    max_features=200,
    ngram_range=(1, 2)
)
tfidf_vocab.fit([all_resume_text])
resume_vocab = list(tfidf_vocab.vocabulary_.keys())
print(f"Resume vocabulary size: {len(resume_vocab)}")
print("Sample terms:", resume_vocab[:30])

Resume vocabulary size: 200
Sample terms: ['information', 'technology', 'specialist', 'systems', 'security', 'responsible', 'managing', 'network', 'support', 'time', 'management', 'strong', 'skills', 'team', 'office', 'organization', 'customer', 'database', 'experience', '01', '2012', '2015', 'company', 'city', 'state', 'implemented', 'reports', 'infrastructure', 'worked', 'computer']


In [13]:
# ── CELL 8: Filter top 1000 Jira tickets ──────────────────────────────────────
# Keep tickets with description longer than JIRA_MIN_DESC_LEN,
# score by keyword overlap with resume vocabulary,
# take the top 1000.

jira_clean = jira_raw.dropna(subset=[JIRA_DESC_COL]).copy()
jira_clean = jira_clean[
    jira_clean[JIRA_DESC_COL].str.len() >= JIRA_MIN_DESC_LEN
].copy()
print(f"Jira after length filter: {len(jira_clean)}")

# Score each ticket against resume vocabulary
jira_clean["match_score"] = jira_clean[JIRA_DESC_COL].apply(
    lambda x: keyword_score(x, resume_vocab)
)

jira_top = (
    jira_clean
    .sort_values("match_score", ascending=False)
    .head(N_JIRA)
    .reset_index(drop=True)
)

# Combine summary and description into a single text field
jira_top["jira_text"] = (
    jira_top[JIRA_SUMMARY_COL].fillna("") + " " +
    jira_top[JIRA_DESC_COL].fillna("")
).str.strip()

print(f"Final Jira pool: {len(jira_top)} tickets")
print(f"Match score range: {jira_top['match_score'].min()} - {jira_top['match_score'].max()}")
display(jira_top[["jira_text", "match_score"]].head(5))

Jira after length filter: 46648
Final Jira pool: 500 tickets
Match score range: 20 - 56


,jira_text,match_score
0,Sourcetree crashes on launch 99% of the time #...,56
1,Sourcetree crashes on launch 99% of the time #...,56
2,Sourcetree crashes on launch 99% of the time #...,56
3,Sourcetree crashes on launch 99% of the time #...,56
4,Sourcetree crashes on launch 99% of the time #...,56


In [14]:
github_raw

,commit,author,date,message,repo
0,692bba578efb5e305c9b116568e5aad75b3fdbb3,Mortada Mehyar <mortada@users.noreply.github.com>,Wed Apr 21 12:27:07 2021 +0800,DOC: add example for plotting asymmetrical err...,pandas-dev/pandas
1,855696cde0ef5d80a7d4bd3f6a2940c5a2fecb3f,Patrick Hoefler <61934744+phofl@users.noreply....,Wed Apr 21 01:23:07 2021 +0200,Add keyword sort to pivot_table (#40954),pandas-dev/pandas
2,eaaefd140289a5103679ac6748567f724c7be56a,attack68 <24256554+attack68@users.noreply.gith...,Wed Apr 21 01:21:22 2021 +0200,ENH: `Styler.highlight_quantile` method (#40926),pandas-dev/pandas
3,aab87997058f3c74ba70286620ebe792ee4ef169,attack68 <24256554+attack68@users.noreply.gith...,Wed Apr 21 01:01:03 2021 +0200,ENH: add `decimal` and `thousands` args to `St...,pandas-dev/pandas
4,9c43cd7675d96174051e470de1f45e2bf7c9ebdc,Simon Hawkins <simonjayhawkins@gmail.com>,Tue Apr 20 23:58:18 2021 +0100,[ArrowStringArray] Use utf8_upper and utf8_low...,pandas-dev/pandas
...,...,...,...,...,...
4336294,43a17862477d0b2b7fb825ffd2a00ac193d2f771,Johannes Rieken <johannes.rieken@gmail.com>,Fri Nov 13 16:36:52 2015 +0100,jsdoc fot vscode.d.ts,microsoft/vscode
4336295,e3281e77cb1b684787971fef7985002583ec843a,Johannes Rieken <johannes.rieken@gmail.com>,Fri Nov 13 15:18:17 2015 +0100,remove commented extension reference,microsoft/vscode
4336296,0a2f0cbc5c7ebc4573ba93c7b4c007efb1110856,Benjamin Pasero <benjpas@microsoft.com>,Fri Nov 13 16:32:42 2015 +0100,gulp-symdest does not preserve links on electr...,microsoft/vscode
4336297,6f9e2ae3907632e2f7dbbabe8da403edd6dfa120,Chris Dias <chris@diasfam.com>,Fri Nov 13 15:48:38 2015 +0100,"Add reference to DefinitelyTyped, updated the ...",microsoft/vscode


In [15]:
# ── CELL 9: Filter top 10000 GitHub commits ───────────────────────────────────
# Score commit messages against resume vocabulary,
# take the top 10000 by match score.

github_clean = github_raw.dropna(subset=[COMMIT_MSG_COL]).copy()

# Drop commits that are just single words or boilerplate
github_clean = github_clean[
    github_clean[COMMIT_MSG_COL].str.len() >= 10
].copy()
print(f"GitHub after length filter: {len(github_clean)}")

# Score each commit against resume vocabulary
# NOTE: this can be slow on a large file — sample first if needed
# If the file is very large (>1M rows), uncomment the line below to pre-sample
# github_clean = github_clean.sample(n=500000, random_state=42)

github_clean["match_score"] = github_clean[COMMIT_MSG_COL].apply(
    lambda x: keyword_score(x, resume_vocab)
)

# Include repo name in commit text for richer signal
github_clean["commit_text"] = (
    github_clean[COMMIT_REPO_COL].fillna("") + " " +
    github_clean[COMMIT_MSG_COL].fillna("")
).str.strip()

github_top = (
    github_clean
    .sort_values("match_score", ascending=False)
    .head(N_COMMITS)
    .reset_index(drop=True)
)

print(f"Final GitHub commit pool: {len(github_top)} commits")
print(f"Match score range: {github_top['match_score'].min()} - {github_top['match_score'].max()}")
display(github_top[["commit_text", "match_score"]].head(5))

GitHub after length filter: 4310538
Final GitHub commit pool: 500 commits
Match score range: 31 - 83


,commit_text,match_score
0,python/cpython Merged revisions 46753-51188 vi...,83
1,tensorflow/tensorflow Merge changes from githu...,81
2,python/cpython Much-needed merge (using svnmer...,77
3,gcc-mirror/gcc Normalise whitespace in GNU Cla...,72
4,"python/cpython Blocked revisions 75914,75932,7...",68


In [16]:
# ── CELL 10: Build composite employee profiles ─────────────────────────────────
# Randomly assign N Jira tickets and M commits to each resume.
# This creates the synthetic "behavioral layer" on top of the resume.
# The profile has three text components:
#   - resume_text  (visible skills — used by baseline)
#   - jira_text    (latent signal 1 — used by AI engine)
#   - commit_text  (latent signal 2 — used by AI engine)

jira_indices = jira_top.index.tolist()
commit_indices = github_top.index.tolist()

profiles = []
for _, row in resumes_sample.iterrows():
    assigned_jira_idx = random.sample(jira_indices, TICKETS_PER_PROFILE)
    assigned_commit_idx = random.sample(commit_indices, COMMITS_PER_PROFILE)

    jira_texts = jira_top.loc[assigned_jira_idx, "jira_text"].tolist()
    commit_texts = github_top.loc[assigned_commit_idx, "commit_text"].tolist()

    profiles.append({
        "employee_id": row["employee_id"],
        "resume_text": str(row[RESUME_TEXT_COL]),
        "jira_text": " | ".join(jira_texts),
        "commit_text": " | ".join(commit_texts),
        # Full profile text used by AI engine
        "full_profile": (
            "RESUME:\n" + str(row[RESUME_TEXT_COL]) + "\n\n" +
            "JIRA TICKETS WORKED ON:\n" + " | ".join(jira_texts) + "\n\n" +
            "GITHUB COMMITS:\n" + " | ".join(commit_texts)
        )
    })

profiles_df = pd.DataFrame(profiles)
print(f"Profiles built: {len(profiles_df)}")
print("\nSample profile preview (truncated):")
print(profiles_df.iloc[0]["full_profile"][:800])

Profiles built: 100

Sample profile preview (truncated):
RESUME:
         INFORMATION TECHNOLOGY SPECIALIST (INFOSEC)       Summary    Retired Information Assurance Systems Security Certification Specialist responsible for managing and monitoring
information systems and network security, and information systems security programs in support of the Information
Security/Information Assurance mission for U.S. Army Medical Command and Defense Health Agency. Also, served as a clerk typist and secretary.      Highlights          Self-directed  Results-oriented  Time management      Strong interpersonal skills  Dedicated team player  Labor relations            Accomplishments     Increased office organization by developing more efficient filing system and customer database protocols.        Experience      INFORMATION TECHNOLOGY SPECIALIST (INFOSEC)   0


In [17]:
# ── CELL 11: Sample 20 job postings ───────────────────────────────────────────
# Filter for postings that have a meaningful description,
# then sample 20.

# Update JOB_TITLE_COL and JOB_DESC_COL in config cell if these are wrong
jobs_clean = job_descriptions.dropna(subset=[JOB_DESC_COL]).copy()
jobs_clean = jobs_clean[
    jobs_clean[JOB_DESC_COL].str.len() >= 100
].copy()
print(f"Job postings after filter: {len(jobs_clean)}")

# Optional: filter for technical postings only to match resume pool
jobs_clean["tech_score"] = jobs_clean[JOB_DESC_COL].apply(
    lambda x: keyword_score(x, TECHNICAL_KEYWORDS)
)
jobs_tech = jobs_clean[jobs_clean["tech_score"] >= 2]
print(f"Technical job postings: {len(jobs_tech)}")

postings_sample = jobs_tech.sample(n=N_POSTINGS, random_state=42).reset_index(drop=True)
postings_sample["posting_id"] = [f"J{i:02d}" for i in range(len(postings_sample))]

print(f"\nSampled {len(postings_sample)} postings")
display(postings_sample[["posting_id", JOB_TITLE_COL]].head(20))

Job postings after filter: 1068
Technical job postings: 837

Sampled 20 postings


,posting_id,Title
0,J00,Cloud DevSecOps Architect
1,J01,Web Developer
2,J02,AR/VR Developer
3,J03,Android Developer Trainee
4,J04,Game Developer
5,J05,Full Stack Developer - Entry Level
6,J06,AI Engineer - Experienced
7,J07,Entry Level Data Scientist
8,J08,JavaScript Developer
9,J09,Frontend Developer - Entry Level


In [18]:
# ── CELL 12: BASELINE — TF-IDF cosine similarity on resume text only ──────────
# For each posting, rank all 100 employee resumes by cosine similarity
# to the job description. Return top-3.
# This represents the status quo: keyword matching on visible skills.

def tfidf_top_k(query_text, candidate_texts, candidate_ids, k=3):
    """
    Vectorize query + candidates with TF-IDF, return top-k candidate IDs
    by cosine similarity.
    """
    all_docs = [query_text] + candidate_texts
    vec = TfidfVectorizer(stop_words="english", ngram_range=(1, 2))
    matrix = vec.fit_transform(all_docs)
    sims = cosine_similarity(matrix[0:1], matrix[1:])[0]
    top_indices = np.argsort(sims)[::-1][:k]
    return [candidate_ids[i] for i in top_indices]

resume_texts = profiles_df["resume_text"].tolist()
employee_ids = profiles_df["employee_id"].tolist()

baseline_results = {}
for _, posting in tqdm(postings_sample.iterrows(), total=len(postings_sample), desc="Baseline"):
    pid = posting["posting_id"]
    jd_text = posting[JOB_DESC_COL]
    top3 = tfidf_top_k(jd_text, resume_texts, employee_ids, k=TOP_K)
    baseline_results[pid] = top3

print("Baseline done.")
for pid, top3 in list(baseline_results.items())[:3]:
    print(f"  {pid}: {top3}")

Baseline: 100%|██████████| 20/20 [00:06<00:00,  2.97it/s]

Baseline done.
  J00: ['E078', 'E068', 'E061']
  J01: ['E032', 'E045', 'E019']
  J02: ['E045', 'E091', 'E003']


In [19]:
# ── Cell 13: PRE-COMPUTE FULL-PROFILE EMBEDDINGS ──────────────────────────────
# Encode all 100 full profiles with text-embedding-3-large.
# The resulting (100 × 3072) matrix is reused by Engines B and C.
# This is the only expensive embedding step — run it once.

EMBED_MODEL = "text-embedding-3-large"
EMBED_DIM   = 3072
MAX_TOKENS  = 8192

def truncate_tokens(text: str, model: str = EMBED_MODEL,
                    max_tokens: int = MAX_TOKENS) -> str:
    enc    = tiktoken.encoding_for_model(model)
    tokens = enc.encode(text)
    return text if len(tokens) <= max_tokens else enc.decode(tokens[:max_tokens])

def batch_embed(texts: list[str], batch_size: int = 10) -> np.ndarray:
    """Embed a list of texts in batches. Returns float32 ndarray (n x dim)."""
    trunc  = [truncate_tokens(t) for t in texts]
    n_cut  = sum(1 for a, b in zip(texts, trunc) if a != b)
    if n_cut:
        print(f"  ⚠  {n_cut} text(s) truncated to {MAX_TOKENS} tokens")
    vecs = []
    for i in range(0, len(trunc), batch_size):
        batch = trunc[i : i + batch_size]
        resp  = openai_client.embeddings.create(input=batch, model=EMBED_MODEL)
        vecs.extend(item.embedding for item in resp.data)
        print(f"  Embedded {min(i+batch_size, len(trunc))}/{len(trunc)}")
    return np.array(vecs, dtype=np.float32)

def embed_one(text: str) -> np.ndarray:
    """Embed a single text. Returns float32 vector."""
    resp = openai_client.embeddings.create(
        input=[truncate_tokens(text)], model=EMBED_MODEL)
    return np.array(resp.data[0].embedding, dtype=np.float32)

def cos_top_k(query_vec: np.ndarray, matrix: np.ndarray,
              ids: list[str], k: int) -> list[str]:
    """Return top-k ids from matrix by cosine similarity to query_vec."""
    sims = cosine_similarity(query_vec.reshape(1, -1), matrix)[0]
    return [ids[i] for i in np.argsort(sims)[::-1][:k]]

print("Encoding 100 full profiles — this runs once and takes ~2 min...")
profile_embeddings = batch_embed(profiles_df["full_profile"].tolist())
print(f"Done. Matrix shape: {profile_embeddings.shape}")


Encoding 100 full profiles — this runs once and takes ~2 min...
  ⚠  100 text(s) truncated to 8192 tokens
  Embedded 10/100
  Embedded 20/100
  Embedded 30/100
  Embedded 40/100
  Embedded 50/100
  Embedded 60/100
  Embedded 70/100
  Embedded 80/100
  Embedded 90/100
  Embedded 100/100
Done. Matrix shape: (100, 3072)


In [20]:
# ── Engine A: CLAUDE ONLY ─────────────────────────────────────────────────────
#
# Step 1 — TF-IDF pre-filter: narrow 100 employees to top-20 by resume-keyword
#           similarity to the job description. (Keeps the Claude prompt feasible.)
# Step 2 — Claude re-ranks the 20 candidates using their FULL profiles
#           (resume + Jira tickets + GitHub commits) and returns the top-3.
#
# Writes: claude_results  {posting_id: [E_id, E_id, E_id]}

CLAUDE_MODEL     = "claude-sonnet-4-20250514"
TFIDF_PREFILTER  = 20   # candidates forwarded to Claude

def claude_rerank(posting_title: str, posting_desc: str,
                  candidates: list[dict], top_k: int = TOP_K) -> list[str]:
    """
    Send a job posting + candidate full-profiles to Claude.
    Each dict in candidates must have keys: employee_id, full_profile.
    Returns up to top_k employee IDs.
    """
    block = "".join(
        f"\n\n--- CANDIDATE {p['employee_id']} ---\n{p['full_profile'][:1200]}"
        for p in candidates
    )
    prompt = (
        f"You are a senior hiring manager at a large financial services firm.\n\n"
        f"Job posting: {posting_title}\n{posting_desc[:800]}\n\n"
        f"Below are {len(candidates)} internal candidate profiles (resume + Jira tickets + GitHub commits).\n"
        f"Identify the top {top_k} candidates who best fit the role based on the FULL profile.\n"
        f"\nCANDIDATES:{block}\n\n"
        f"Return ONLY a JSON array of the top {top_k} employee IDs, e.g. [\"E001\",\"E045\",\"E023\"].\n"
        f"Nothing else."
    )
    resp = anthropic_client.messages.create(
        model=CLAUDE_MODEL, max_tokens=100,
        messages=[{"role": "user", "content": prompt}]
    )
    raw = resp.content[0].text.strip()
    try:
        return json.loads(raw)[:top_k]
    except json.JSONDecodeError:
        return re.findall(r'E\d{3}', raw)[:top_k]

# Build quick lookup: employee_id → full_profile
profile_lookup = profiles_df.set_index("employee_id").to_dict(orient="index")

resume_texts = profiles_df["resume_text"].tolist()
employee_ids = profiles_df["employee_id"].tolist()

claude_results = {}
for _, posting in tqdm(postings_sample.iterrows(),
                        total=len(postings_sample), desc="Engine A — Claude Only"):
    pid   = posting["posting_id"]
    jd    = posting[JOB_DESC_COL]
    title = posting[JOB_TITLE_COL]

    # TF-IDF pre-filter
    pre_ids    = tfidf_top_k(jd, resume_texts, employee_ids, k=TFIDF_PREFILTER)
    candidates = [{"employee_id": eid,
                   "full_profile": profile_lookup[eid]["full_profile"]}
                  for eid in pre_ids]
    claude_results[pid] = claude_rerank(title, jd, candidates, top_k=TOP_K)
    time.sleep(0.4)

print("Engine A done.")
for pid, top3 in list(claude_results.items())[:3]:
    print(f"  {pid}: {top3}")


Engine A — Claude Only:   0%|          | 0/20 [00:00<?, ?it/s]/tmp/ipykernel_8616/2805265527.py:33: DeprecationWarning: The model 'claude-sonnet-4-20250514' is deprecated and will reach end-of-life on June 15th, 2026.
Please migrate to a newer model. Visit https://docs.anthropic.com/en/docs/resources/model-deprecations for more information.
  resp = anthropic_client.messages.create(
Engine A — Claude Only: 100%|██████████| 20/20 [02:30<00:00,  7.50s/it]

Engine A done.
  J00: ['E078', 'E031', 'E077']
  J01: ['E032', 'E019', 'E092']
  J02: ['E089', 'E032', 'E086']


In [21]:
# ── Engine B: EMBEDDINGS ONLY ─────────────────────────────────────────────────
#
# Embed each job description, then rank all 100 employees by cosine similarity
# to their pre-computed full-profile embedding. No LLM reasoning involved.
#
# Writes: embedding_results  {posting_id: [E_id, E_id, E_id]}

embedding_results = {}
for _, posting in tqdm(postings_sample.iterrows(),
                        total=len(postings_sample), desc="Engine B — Embeddings Only"):
    pid    = posting["posting_id"]
    jd_vec = embed_one(posting[JOB_DESC_COL])
    embedding_results[pid] = cos_top_k(jd_vec, profile_embeddings,
                                        employee_ids, k=TOP_K)

print("Engine B done.")
for pid, top3 in list(embedding_results.items())[:3]:
    print(f"  {pid}: {top3}")


Engine B — Embeddings Only: 100%|██████████| 20/20 [00:05<00:00,  3.57it/s]

Engine B done.
  J00: ['E035', 'E019', 'E077']
  J01: ['E098', 'E078', 'E074']
  J02: ['E098', 'E091', 'E055']


In [22]:
# ── Engine C: SHORTLIST + CLAUDE ─────────────────────────────────────────────
#
# Step 1 — Embedding pre-filter: rank all 100 employees by cosine similarity
#           to the job-description embedding; keep top-20.
#           (Semantically better shortlist than TF-IDF keyword matching.)
# Step 2 — Claude re-ranks the top-20 using their FULL profiles, same prompt
#           as Engine A.
#
# Writes: shortlist_claude_results  {posting_id: [E_id, E_id, E_id]}

EMBED_PREFILTER = 20   # candidates forwarded to Claude

shortlist_claude_results = {}
for _, posting in tqdm(postings_sample.iterrows(),
                        total=len(postings_sample),
                        desc="Engine C — Shortlist + Claude"):
    pid   = posting["posting_id"]
    jd    = posting[JOB_DESC_COL]
    title = posting[JOB_TITLE_COL]

    # Embedding pre-filter: semantically relevant shortlist
    jd_vec  = embed_one(jd)
    pre_ids = cos_top_k(jd_vec, profile_embeddings, employee_ids, k=EMBED_PREFILTER)

    candidates = [{"employee_id": eid,
                   "full_profile": profile_lookup[eid]["full_profile"]}
                  for eid in pre_ids]
    shortlist_claude_results[pid] = claude_rerank(title, jd, candidates, top_k=TOP_K)
    time.sleep(0.4)

print("Engine C done.")
for pid, top3 in list(shortlist_claude_results.items())[:3]:
    print(f"  {pid}: {top3}")


Engine C — Shortlist + Claude:   0%|          | 0/20 [00:00<?, ?it/s]/tmp/ipykernel_8616/2805265527.py:33: DeprecationWarning: The model 'claude-sonnet-4-20250514' is deprecated and will reach end-of-life on June 15th, 2026.
Please migrate to a newer model. Visit https://docs.anthropic.com/en/docs/resources/model-deprecations for more information.
  resp = anthropic_client.messages.create(
Engine C — Shortlist + Claude:   5%|▌         | 1/20 [00:02<00:42,  2.24s/it]/tmp/ipykernel_8616/2805265527.py:33: DeprecationWarning: The model 'claude-sonnet-4-20250514' is deprecated and will reach end-of-life on June 15th, 2026.
Please migrate to a newer model. Visit https://docs.anthropic.com/en/docs/resources/model-deprecations for more information.
  resp = anthropic_client.messages.create(
Engine C — Shortlist + Claude:  10%|█         | 2/20 [00:16<02:49,  9.42s/it]/tmp/ipykernel_8616/2805265527.py:33: DeprecationWarning: The model 'claude-sonnet-4-20250514' is deprecated and will reach end-o

Engine C done.
  J00: ['E031', 'E081', 'E002']
  J01: ['E062', 'E025', 'E069']
  J02: ['E093', 'E047', 'E004']


In [25]:
# ── CELL 16: GROUND TRUTH — Claude reads resume vs posting ────────────────────
# Claude acts as a senior hiring manager.
# It reads the job posting + resume text only (not behavioral signals)
# and independently labels the top-3 best fits.
#
# This call is independent of both baseline and AI engine.
# Results are cached to disk so you do not re-run and pay API costs.
#
# To use manual labels instead:
#   ground_truth = {"J00": ["E005", "E032", "E071"], "J01": [...], ...}
#   and skip this cell.


# Initialize the client
# It's best practice to set your API key as an environment variable
client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))

GT_CACHE_FILE = "ground_truth_labels.json"

def claude_ground_truth(posting_title, posting_desc, resume_candidates, top_k=3):
    candidates_block = ""
    for p in resume_candidates:
        candidates_block += f"\n--- {p['employee_id']} ---\n{p['resume_text'][:600]}\n"

    prompt = (
        f"You are a senior hiring manager.\n\n"
        f"Job posting: {posting_title}\n{posting_desc[:500]}\n\n"
        f"Candidate resumes:{candidates_block}\n"
        f"Return ONLY a JSON array of the top {top_k} employee IDs who best fit this role. "
        f"Example: [\"E001\", \"E045\", \"E023\"]. Nothing else."
    )
    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=80,
        messages=[{"role": "user", "content": prompt}]
    )
    raw = response.content[0].text.strip()
    try:
        return json.loads(raw)[:top_k]
    except json.JSONDecodeError:
        return re.findall(r'E\d{3}', raw)[:top_k]

# Load cache if it exists
if os.path.exists(GT_CACHE_FILE):
    with open(GT_CACHE_FILE) as f:
        ground_truth = json.load(f)
    print(f"Ground truth loaded from cache ({len(ground_truth)} postings)")
else:
    resume_candidates = [
        {"employee_id": row["employee_id"], "resume_text": row["resume_text"]}
        for _, row in profiles_df.iterrows()
    ]
    ground_truth = {}
    for _, posting in tqdm(postings_sample.iterrows(), total=len(postings_sample), desc="Ground Truth"):
        pid = posting["posting_id"]
        gt = claude_ground_truth(
            posting[JOB_TITLE_COL],
            posting[JOB_DESC_COL],
            resume_candidates,
            top_k=TOP_K
        )
        ground_truth[pid] = gt
        time.sleep(0.3)

    with open(GT_CACHE_FILE, "w") as f:
        json.dump(ground_truth, f, indent=2)
    print(f"Ground truth generated and cached to {GT_CACHE_FILE}")

for pid, gt in list(ground_truth.items())[:3]:
    print(f"  {pid}: {gt}")

Ground Truth:   0%|          | 0/20 [00:00<?, ?it/s]/tmp/ipykernel_8616/557234091.py:32: DeprecationWarning: The model 'claude-sonnet-4-20250514' is deprecated and will reach end-of-life on June 15th, 2026.
Please migrate to a newer model. Visit https://docs.anthropic.com/en/docs/resources/model-deprecations for more information.
  response = client.messages.create(
Ground Truth: 100%|██████████| 20/20 [07:27<00:00, 22.35s/it]

Ground truth generated and cached to ground_truth_labels.json
  J00: ['E035', 'E053', 'E081']
  J01: ['E019', 'E058', 'E082']
  J02: ['E047', 'E023', 'E011']


In [26]:
# ── Precision@3 — All Four Engines ───────────────────────────────────────────

def precision_at_k(predicted: list, gt_set: set, k: int = TOP_K) -> float:
    return sum(1 for p in predicted[:k] if p in gt_set) / k

rows = []
for _, posting in postings_sample.iterrows():
    pid   = posting["posting_id"]
    title = posting[JOB_TITLE_COL]
    gt    = set(ground_truth.get(pid, []))

    bl = baseline_results.get(pid, [])
    ca = claude_results.get(pid, [])
    em = embedding_results.get(pid, [])
    sc = shortlist_claude_results.get(pid, [])

    rows.append({
        "posting_id":          pid,
        "job_title":           title,
        "ground_truth":        list(gt),
        # top-3 picks per engine
        "baseline_top3":       bl,
        "claude_top3":         ca,
        "embedding_top3":      em,
        "shortlist_claude_top3": sc,
        # precision scores
        "p3_baseline":         round(precision_at_k(bl, gt), 4),
        "p3_claude":           round(precision_at_k(ca, gt), 4),
        "p3_embedding":        round(precision_at_k(em, gt), 4),
        "p3_shortlist_claude": round(precision_at_k(sc, gt), 4),
    })

results_df = pd.DataFrame(rows)

# Lift columns (delta vs baseline)
for eng in ["claude", "embedding", "shortlist_claude"]:
    results_df[f"lift_{eng}"] = (
        results_df[f"p3_{eng}"] - results_df["p3_baseline"]
    )

avg_bl = results_df["p3_baseline"].mean()
avg_ca = results_df["p3_claude"].mean()
avg_em = results_df["p3_embedding"].mean()
avg_sc = results_df["p3_shortlist_claude"].mean()

# ── Per-posting table ─────────────────────────────────────────────────────────
print("=" * 80)
print(f"{'PRECISION@3 — PER POSTING':^80}")
print("=" * 80)
display(results_df[[
    "posting_id", "job_title",
    "p3_baseline", "p3_claude", "p3_embedding", "p3_shortlist_claude"
]])

# ── Summary table ─────────────────────────────────────────────────────────────
print("=" * 80)
print(f"{'SUMMARY':^80}")
print("=" * 80)
print(f"{'Engine':<32} {'Avg P@3':>8}  {'Lift vs Baseline':>18}")
print("-" * 62)
print(f"{'Baseline  (TF-IDF, resume only)':<32} {avg_bl:>8.4f}  {'—':>18}")
print(f"{'Engine A  (Claude Only)':<32} {avg_ca:>8.4f}  {avg_ca - avg_bl:>+17.4f} pp")
print(f"{'Engine B  (Embeddings Only)':<32} {avg_em:>8.4f}  {avg_em - avg_bl:>+17.4f} pp")
print(f"{'Engine C  (Shortlist + Claude)':<32} {avg_sc:>8.4f}  {avg_sc - avg_bl:>+17.4f} pp")
print("=" * 62)


                           PRECISION@3 — PER POSTING                            


,posting_id,job_title,p3_baseline,p3_claude,p3_embedding,p3_shortlist_claude
0,J00,Cloud DevSecOps Architect,0.0000,0.0000,0.3333,0.3333
1,J01,Web Developer,0.3333,0.3333,0.0000,0.0000
2,J02,AR/VR Developer,0.0000,0.0000,0.0000,0.3333
3,J03,Android Developer Trainee,0.3333,0.3333,0.0000,0.0000
4,J04,Game Developer,0.0000,0.0000,0.0000,0.3333
5,J05,Full Stack Developer - Entry Level,0.6667,0.3333,0.0000,0.0000
6,J06,AI Engineer - Experienced,0.0000,0.6667,0.0000,0.3333
7,J07,Entry Level Data Scientist,0.3333,0.6667,0.0000,0.6667
8,J08,JavaScript Developer,0.0000,0.6667,0.0000,0.0000
9,J09,Frontend Developer - Entry Level,0.3333,0.6667,0.0000,0.0000


                                    SUMMARY                                     
Engine                            Avg P@3    Lift vs Baseline
--------------------------------------------------------------
Baseline  (TF-IDF, resume only)    0.2167                   —
Engine A  (Claude Only)            0.4000            +0.1833 pp
Engine B  (Embeddings Only)        0.0333            -0.1833 pp
Engine C  (Shortlist + Claude)     0.1333            -0.0833 pp


In [27]:
# ── Save Results ──────────────────────────────────────────────────────────────

results_df.to_csv("experiment_results.csv", index=False)

summary = {
    "embed_model":          EMBED_MODEL,
    "claude_model":         CLAUDE_MODEL,
    "ground_truth_method":  "Claude as simulated hiring manager (resume-only view, cached)",
    "n_employees":          len(profiles_df),
    "n_postings":           N_POSTINGS,
    "tickets_per_profile":  TICKETS_PER_PROFILE,
    "commits_per_profile":  COMMITS_PER_PROFILE,
    "top_k":                TOP_K,
    "tfidf_prefilter_k":    TFIDF_PREFILTER,
    "embed_prefilter_k":    EMBED_PREFILTER,
    "results": {
        "baseline":         {"avg_p3_pct": round(avg_bl * 100, 1)},
        "claude_only":      {"avg_p3_pct": round(avg_ca * 100, 1),
                             "lift_pp":    round((avg_ca - avg_bl) * 100, 1)},
        "embeddings_only":  {"avg_p3_pct": round(avg_em * 100, 1),
                             "lift_pp":    round((avg_em - avg_bl) * 100, 1)},
        "shortlist_claude": {"avg_p3_pct": round(avg_sc * 100, 1),
                             "lift_pp":    round((avg_sc - avg_bl) * 100, 1)},
    },
}

with open("experiment_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))
print("\nSaved: experiment_results.csv, experiment_summary.json")


{
  "embed_model": "text-embedding-3-large",
  "claude_model": "claude-sonnet-4-20250514",
  "ground_truth_method": "Claude as simulated hiring manager (resume-only view, cached)",
  "n_employees": 100,
  "n_postings": 20,
  "tickets_per_profile": 5,
  "commits_per_profile": 10,
  "top_k": 3,
  "tfidf_prefilter_k": 20,
  "embed_prefilter_k": 20,
  "results": {
    "baseline": {
      "avg_p3_pct": 21.7
    },
    "claude_only": {
      "avg_p3_pct": 40.0,
      "lift_pp": 18.3
    },
    "embeddings_only": {
      "avg_p3_pct": 3.3,
      "lift_pp": -18.3
    },
    "shortlist_claude": {
      "avg_p3_pct": 13.3,
      "lift_pp": -8.3
    }
  }
}

Saved: experiment_results.csv, experiment_summary.json
